<a href="https://colab.research.google.com/github/davidlealo/practicos_sisrec_2026/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Interrogación 1 - Métricas de Recomendación



## Instrucciones Generales

1. Está prohibido el uso de IA como Chat GPT, Gemini, Claude o similares. Si se sorprende a alguien usándolo se le pedirá la prueba y se evaluará con 1,1.
2. En caso de utilizar Google Colab deben deshabilitar la *Asistencia con IA* (Ajustes > Asistencia con IA > Mostrar opciones de autocompletado de códigos basados en IA / Con permiso para usar funciones de IA generativa). Para quienes utilicen un IDE deben desactivar el autocompletado de herramientas de IA (Copilot, Claude Code, Codex, etc.).
3. Pueden realizar el práctico tanto en Google Colab como en un Jupyter Notebook local, pero no olvide entregar antes de que termine la hora de clases en este buzón de Canvas. Debe entregar únicamente el archivo main.ipynb completo.
4. Esta actividad es individual, está prohibido conversar con compañeros para realizarla.
5. Sí está permitido el uso de código de los prácticos realizados en las ayudantías, tanto el que se te entrega por el equipo docente como el que tú mismo/a has realizado.

Para realizar este práctico solo es necesario tener instalado Numpy y Pandas. Se sugiere tener instalada la última versión de Python y de Numpy.

La evaluación constará de una parte teórica y una segunda parte práctica donde deberás implementar con código en Python la métrica solicitada.

**NOMBRE ESTUDIANTE**: David Leal Olivares

## Preguntas Teóricas (3pts)

En esta sección deberás contestar 3 preguntas teóricas

1. Describe la intuición detrás de la métrica **MRR (Mean Reciprocal Rank)** y **cómo se calcula**. ¿Qué tipo de tarea de recomendación evalúa mejor (por ejemplo, búsqueda, recomendación de un "ítem único", listas largas)? ¿Qué limitación tiene MRR para evaluar una la lista de recomendaciones?

**Respuesta**: MRR es una métrica que busca calcular el peso del primer elemento relevante encontrado en una lista de recomendación. Su fórmula es:

$MRR = \frac1r$ donde r es el ranking del primer elemento relavante encontrado.

Esta es una métrica que puede servir principalmente para premiar el encontrar lo más rápido posible los elementos relevantes, por ejemplo, para un usuario que en sistemas de consumo rápido o en pantallas pequeñas, como dispositivos móviles, necesitemos mostrar rápidamente buenas recomendaciones a los usuarios.

Obviamente va a evaluar mejor cuando las listas son más pequeñas, y esa puede ser su limitante, porque va a evaluar de la misma manera a dos listados que encuentran en segundo o tercer puesto a un ítem relevante independiente de la cantidad de ítems y la cantidad de relevantes en el conjunto de datos antes mencionado.

2. Explica qué miden la **Diversidad (ILS de Ziegler)** y la **Novedad (self-information)**, cómo se calculan, y por qué son consideradas métricas *complementarias* a las métricas de accuracy (Precision, nDCG, etc.). ¿**Cómo afectan la Diversidad y la Novedad la percepción del usuario** sobre la calidad del recomendador (satisfacción, descubrimiento de nuevos ítems, efecto "filter bubble")? Da un ejemplo de un recomendador con alto nDCG pero baja novedad/diversidad y discute qué podría percibir el usuario frente a ese sistema.

**Respuesta**: La Diversidad (ILS de Ziegler) se calcula comparando la similaridad entre los pares de elementos recomendados. Existen mediciones que afirman que los usuarios tienen mayor satisfacción no solamente por una recomendación más precisa, sino también por la diversidad, para navegar en contenidos que no aburran al usuario.

La métrica de Novedad (self-information) se calcula como el logaritmo inverso de la popularidad del ítem, lo cual permite al usuario no solamente obtener buenas recomendaciones, sino algunas que sin ser las más populares se acerque a sus gustos.

Estas métricas son complementarias a las de raking porque aunque no cambian los ítems mejores para el usuario, si pueden destacar en el ranking entregado algunos ítems que permiten elevar la satisfacción de la muestra. Pensemos que yo veo películas de vampiros, tal vez lo más fácil es seguir recomendando del mismo género, pero eso puede aburrir a un usuario, entonces al entregar diversidad entre los límites de lo esperado, el usuario puede sin afectar su raking conocer más contenidos y valorar más la aplicación, como en el ejemplo que es un servicio de películas, rompiendo un poco la burbuja de información. Con la novedad es lo mismo, eso ayuda al usuario a conocer nuevos ítems, autores, etc.

Por ejemplo podríamos tener un nDCG perfecto, muy similar al iDCG, pero con cero novedad y diversidad, que meta al usuario en una burbuja de información o en una saturación de contenido igual al consumido, como pasa en redes sociales, sin contribuir al que el usuario conozca más opiniones sobre un tema, siguiendo con el ejemplo de redes sociales.

3. Explica con tus palabras en qué se diferencian **Precision** y **Recall** en el contexto de sistemas de recomendación, **tanto en la forma en que se calculan como en la forma de interpretar su resultado**. Luego, explica qué aporta **P@N (Precision at N)** respecto a la Precision global. ¿Por qué P@N suele ser más informativa al evaluar un ranker? Da un ejemplo donde dos sistemas tengan la misma Precision global pero un P@10 muy distinto.

**Respuesta:** La precisión es la métrica calcula los ítems recomendados y relevantes al mismo tiempo entre mis recomendados y el recall es la que evalúa los ítems recomendados y relevantes al mismo tiempo entre los revelantes.

Entonces acá existe un problema que muestra la curva precision-recall. Para obtener una mayor precisión yo debo ser capaz de recomendar lo antes posible la mayor cantidad de ítems relevantes, un ejemplo de precisión 1 es cuando el primer elemento ya es relavante de una lista de un elemento, en cambio el recall al dividir por los relevantes, si yo hago una predicción de todos los elementos de la lista el recall va a tender a 1, porque entre todos los elementos de una lista, obviamente van a estar los relevantes. Comúnmente la curva precision-recall tiende a bajar el precision en la medida que el recall sube.

Entonces al limitar la precision por ejemplo con un `P@N`estamos fomentando que el recomendador tenga que mejorar su performance para generar lo antes posible la mejor lista de ranking posible.

## Implementación de Métrica: Zero Proportion at k (6 pts)

En esta sección deberás implementar la métrica **Zero Proportion @ k**, la cual mide la **fracción de usuarios para los cuales el recomendador no logra posicionar ningún ítem relevante dentro del top-k** de su lista de recomendaciones. Es una métrica útil para diagnosticar "fallos totales" del sistema (usuarios a los que no se les recomienda nada útil) y complementa a las métricas de ranking como AUC, nDCG o MAP.

Para hacer la evaluación vamos a utilizar el algoritmo de *collaborative filtering* denominado **Sapling Similarity**, presentado en el paper [Sapling Similarity: a performing and interpretable memory-based tool for recommendation](https://arxiv.org/abs/2210.07039), aplicado al dataset MovieLens 100K. Este dataset contiene 100.000 tuplas (usuario, item, ranking), de las cuales 80.000 se utilizarán para entrenamiento y 20.000 para testear. Los rankings se encuentran en un rango de 1 a 5.

Contarán con dos instancias del modelo ya entrenados: Basada en Usuario y Basada en Items. No deben modificar el código del modelo ni del entrenamiento.

In [50]:
from sapling_similarity import SaplingSimilarity
import numpy as np

In [51]:
# modelo basado en usuarios
sapling_similarity_users_model = SaplingSimilarity.from_csv(
    train_csv="train.csv",
    test_csv="test.csv",
    user_column_name="user_id",
    item_column_name="movie_id",
    rating_column_name="rating",
    threshold=3,
    based_on="user"
)

sapling_similarity_users_model.train()

# modelo basado en items
sapling_similarity_items_model = SaplingSimilarity.from_csv(
    train_csv="train.csv",
    test_csv="test.csv",
    user_column_name="user_id",
    item_column_name="movie_id",
    rating_column_name="rating",
    threshold=3,
    based_on="item"
)

sapling_similarity_items_model.train()

/content/sapling_similarity.py:129: RuntimeWarning: invalid value encountered in divide
  B = np.nan_to_num((1 - (co_ocurrences_matrix * (1 - co_ocurrences_matrix / items_interactions) + (items_interactions - co_ocurrences_matrix.T).T * (1 - (items_interactions - co_ocurrences_matrix.T).T / (
/content/sapling_similarity.py:130: RuntimeWarning: invalid value encountered in divide
  number_of_items - items_interactions))).T / (items_interactions * (1 - items_interactions / number_of_items))).T * np.sign(((co_ocurrences_matrix * number_of_items / items_interactions).T / items_interactions).T - 1))


Los modelos entrenados cuenta con el método `test`, el cual retorna dos elementos:
- El primer elemento corresponde a una lista de tuplas con los **ratings reales** para las combinaciones usuarios e items presentes en el set de testeo, con la forma (user_id, item_id, rating_real).

- El segundo elemento corresponde a una lista de tuplas con los **ratings predichos** por el algoritmo para las combinaciones usuarios e items presentes en el set de testeo, con la forma (user_id, item_id, rating_predicted).

In [52]:
y_test, y_pred_users = sapling_similarity_users_model.test()
y_test, y_pred_items = sapling_similarity_items_model.test()

Además, los modelos entrenados cuentan con el método `predict`, que retorna para un `user_id` específico las lista de tuplas (item_id, ranking) de los items que el usuario no ha evaluado. Con estas predicciones es posible rankear una lista de items para un usuario ordenando descendente los items según el rating que predice el modelo.

In [53]:
sapling_similarity_users_model.predict(user_id=1)[40:50]

[(1, np.int64(613), np.float64(4.300253965152188)),
 (1, np.int64(285), np.float64(4.293830614167815)),
 (1, np.int64(173), np.float64(4.290822452377635)),
 (1, np.int64(641), np.float64(4.2713802820914895)),
 (1, np.int64(1111), np.float64(4.26898936283368)),
 (1, np.int64(357), np.float64(4.265864353132638)),
 (1, np.int64(513), np.float64(4.262441584025546)),
 (1, np.int64(1431), np.float64(4.26102251250427)),
 (1, np.int64(657), np.float64(4.261015377303021)),
 (1, np.int64(479), np.float64(4.260759452916226))]

Se les entrega un diccionario con las predicciones hechas por cada una de las versiones del algoritmo, donde cada llave es un user_id y tiene como valor una lista con tuplas (item, rating), donde rating es el valor que el modelo predice que el usuario asignará al item. Los elementos para cada usuario estan ordenadas por el rating que predijo el modelo.

In [54]:
user_based_predictions = {user_id: sapling_similarity_users_model.predict(user_id) for user_id in sapling_similarity_users_model.user_map.keys()}
item_based_predictions = {user_id: sapling_similarity_items_model.predict(user_id) for user_id in sapling_similarity_items_model.user_map.keys()}

1. Crea un diccionario con los items relevantes para cada usuario presentes en `y_test`. Cada llave del diccionario será un user_id y tendrá como valor un lista que contenga los item_ids de los items relevantes para ese usuario. Considere como relevante solo los items que se les asignó un **rating mayor o igual a 3**.

Ejemplo:
```python
{
  1: [3, 4, 8,...],
  2: [23, 24, 53,...],
  ...
}
```
donde [3, 4, 8,...] son los items relevantes para el usuario con ID = 1.

In [60]:
user_relevant_items = {}
for user_id, item_id, rating in y_test:
    if user_id not in user_relevant_items:
        user_relevant_items[user_id] = []
    if rating >= 3:
        user_relevant_items[user_id].append(item_id)

2. Completa la función `user_zero_hit`, la cual recibe como parámetros:

- `user_recommendations`: Lista de tuplas (item_id, rating_predicted) de un usuario, **ordenada descendentemente por rating predicho**.

- `user_relevant_items`: Lista de item_ids relevantes para el usuario.

- `k`: Largo de la lista de recomendación a considerar (top-k).

Esta función debe retornar **1** si **ninguno** de los primeros $k$ ítems recomendados se encuentra entre los relevantes del usuario, y **0** si al menos uno lo está. Formalmente:

$$
\text{zero_hit}(u, k) =
\begin{cases}
1 & \text{si } \{i_1, i_2, \dots, i_k\} \cap I_u = \emptyset \\
0 & \text{en otro caso}
\end{cases}
$$

donde $\{i_1, \dots, i_k\}$ son los top-k ítems recomendados al usuario $u$ e $I_u$ es el conjunto de ítems relevantes para $u$.

---

**Ejemplo**

Supongamos que para un usuario tenemos:

`user_relevant_items = [A, B]`

`user_recommendations = [(C, 4.5), (D, 4.4), (E, 3.9), (A, 2.6)]`

- Con `k = 3` → top-3 = [C, D, E], intersección con [A, B] = ∅ → `zero_hit = 1` (fallo total en top-3).
- Con `k = 4` → top-4 = [C, D, E, A], intersección con [A, B] = {A} → `zero_hit = 0`.

In [61]:
def user_zero_hit(user_recommendations, user_relevant_items, k):
    # 1. Cortamos al top K
    top_k = user_recommendations[:k]

    # 2. El truco: DESEMPAQUETAR la tupla directamente en el for
    for item_id, rating_predicho in top_k:

        # 3. Solo comparamos el ID
        if item_id in user_relevant_items:
            # Encontramos al menos uno bueno. ¡No es un zero hit!
            return 0

    # Si el ciclo termina y no encontró nada, es un fallo total
    return 1

3. Llama a la función `zero_proportion` para cada modelo y compara usando valores de k de 10, 100, 500, 1.000 y 2.000. Analice los resultados y mencione por qué podría producirse la diferencia de valores entre ambos modelos y entre distintos valores de k.

In [62]:
def zero_proportion(predictions_dict, k):
    """
    Calcula la proporción de usuarios cuyo top-k no contiene ningún ítem relevante,
    dado un diccionario de predicciones y un tamaño k de recomendaciones.
    """
    zero_hits = []

    for user_id, predictions in predictions_dict.items():
        if user_id in user_relevant_items:
            # Convertir tripletas (user_id, item_id, rating) a pares (item_id, rating)
            rec_list = []
            for _, item_id, rating in predictions:
                rec_list.append((int(item_id), float(rating)))

            zero_hits.append(user_zero_hit(rec_list, user_relevant_items[user_id], k))

    return sum(zero_hits) / len(zero_hits) if zero_hits else 0.0


In [64]:
# El arreglo de Ks que te dieron
ks = [10, 100, 500, 1000, 2000]

# El ciclo para imprimir y comparar
for k in ks:
    zp_user = zero_proportion(user_based_predictions, k)
    zp_item = zero_proportion(item_based_predictions, k)
    print(f"K={k} | User-based: {zp_user:.4f} | Item-based: {zp_item:.4f}")

K=10 | User-based: 0.9877 | Item-based: 0.6187
K=100 | User-based: 0.1822 | Item-based: 0.1761
K=500 | User-based: 0.0123 | Item-based: 0.0138
K=1000 | User-based: 0.0046 | Item-based: 0.0046
K=2000 | User-based: 0.0046 | Item-based: 0.0046


**Respuesta**: -Coloque su respuesta en esta celda-

## Implementación de Métrica: nDCG con Relevancia Graduada (9 pts)

En esta sección deberás implementar la métrica **nDCG@k (Normalized Discounted Cumulative Gain)** usando los mismos modelos Sapling Similarity (user-based e item-based) sobre MovieLens 100K. A diferencia del nDCG clásico que trabaja con relevancia binaria, aquí se usará **relevancia graduada** derivada del rating real del set de test con el siguiente mapeo:

| rating real | $\text{rel}_i$ |
|---:|---:|
| < 3 | 0 |
| 3 | 1 |
| 4 | 2 |
| 5 | 3 |

De esta forma el valor de $\text{rel}_i$ en la fórmula de nDCG tomará valores en $\{0, 1, 2, 3\}$, lo que permite premiar más fuertemente el acierto de ítems que el usuario evaluó con rating alto (5) que los que evaluó con rating apenas positivo (3).

La fórmula para nDCG es la siguiente:

$$\text{DCG@k} = \sum_{i=1}^{k} \frac{\text{rel}_i}{\log_2(i+1)} \qquad \text{nDCG@k} = \frac{\text{DCG@k}}{\text{iDCG@k}}$$

donde $\text{iDCG@k}$ es el DCG del ranking ideal.

1. Crea un diccionario `user_graded_relevance` con los ítems relevantes **graduados** para cada usuario presentes en `y_test`. Cada llave del diccionario será un `user_id` y tendrá como valor otro diccionario `{item_id: rel_grade}`, donde `rel_grade` es la relevancia graduada según la tabla anterior (solo se incluyen ítems con rating ≥ 3, es decir relevancia > 0).

Ejemplo:
```python
{
  1: {3: 2, 4: 3, 8: 1, ...},
  2: {23: 3, 24: 1, 53: 2, ...},
  ...
}
```
donde el usuario 1 evaluó al ítem 3 con rating 4 (rel = 2), al ítem 4 con rating 5 (rel = 3) y al ítem 8 con rating 3 (rel = 1).

In [65]:
user_graded_relevance = {}

for user_id, item_id, rating in y_test:
    # 1. Creamos el diccionario anidado si el usuario no existe
    if user_id not in user_graded_relevance:
        user_graded_relevance[user_id] = {}

    # 2. Solo guardamos si la nota es >= 3 (Relevancia > 0)
    if rating >= 3:
        # Matemáticas simples: 3-2=1, 4-2=2, 5-2=3. ¡Nos ahorramos 4 ifs!
        grado = int(rating - 2)
        user_graded_relevance[user_id][item_id] = grado

2. Completa la función `dcg_at_k`, la cual recibe como parámetros:

- `relevances`: Lista de **relevancias graduadas** (valores enteros en $\{0, 1, 2, 3\}$) de los ítems **en el mismo orden en que fueron rankeados por el modelo**. Es decir, `relevances[0]` es la relevancia graduada del ítem que el modelo puso en la posición 1 del ranking, `relevances[1]` la del ítem en posición 2, y así sucesivamente. Si el ítem de la posición $i$ no es relevante para el usuario (rating < 3 o no está en `user_graded_relevance`), en esa posición debe ir un 0.

    Ejemplo: si el modelo recomienda los ítems `[C, D, E, A, F]` en ese orden, y el usuario tiene `user_graded_relevance = {A: 3, B: 1, E: 2}`, entonces la lista a pasar es:
    ```python
    relevances = [0, 0, 2, 3, 0]
    ```
    (C no está → 0, D no está → 0, E tiene rel=2, A tiene rel=3, F no está → 0).

- `k`: Largo del top-k a considerar. La función debe usar únicamente las primeras $k$ posiciones de `relevances`.

La función debe retornar el DCG@k, definido como:

$$\text{DCG@k} = \sum_{i=1}^{k} \frac{\text{rel}_i}{\log_2(i+1)}$$

donde $\text{rel}_i$ es la relevancia graduada en la posición $i$ del ranking (la posición 1 es la primera posición, por lo tanto el primer término tiene denominador $\log_2(2) = 1$).

In [66]:
import math

def dcg_at_k(relevances, k):
    dcg = 0.0
    # Cortamos la lista a tamaño k y usamos enumerate
    for i, relevancia in enumerate(relevances[:k]):
        # La fórmula pide log2(i + 1), pero como el índice de Python
        # parte en 0, debemos sumar 2.
        dcg += relevancia / math.log2(i + 2)
    return dcg


3. Completa la función `user_ndcg`, la cual recibe como parámetros:

- `user_recommendations`: Lista de tuplas `(item_id, rating_predicted)` del usuario, **ordenada descendentemente por rating predicho**.
- `user_graded_relevance`: Diccionario `{item_id: rel_grade}` con las relevancias graduadas del usuario.
- `k`: Largo del top-k a considerar.

La función debe construir la secuencia de relevancias graduadas del top-k del ranking, calcular el DCG@k, calcular el iDCG@k, y retornar $\text{nDCG@k} = \text{DCG@k} / \text{iDCG@k}$.

In [67]:
def user_ndcg(user_recommendations, user_graded_relevance, k):

    # 1. Construir la lista de relevancias en el orden que recomendó el modelo
    relevancias_reales = []

    for item_id, pred_rating in user_recommendations[:k]:
        # Vamos al diccionario a buscar qué nota le puso el usuario a esta peli.
        # Si no la ha visto (no está en el diccionario), asumimos que es 0.
        rel_grado = user_graded_relevance.get(item_id, 0)
        relevancias_reales.append(rel_grado)

    # 2. Calculamos el DCG Real
    dcg_real = dcg_at_k(relevancias_reales, k)

    # 3. Construimos el "Mundo Ideal" (iDCG)
    # Tomamos todas las notas que este usuario le ha puesto a las películas,
    # y las ordenamos de mayor a menor.
    todas_sus_notas = list(user_graded_relevance.values())
    relevancias_ideales = sorted(todas_sus_notas, reverse=True)

    idcg = dcg_at_k(relevancias_ideales, k)

    # 4. Prevención de división por cero y retorno
    if idcg == 0:
        return 0.0

    return dcg_real / idcg

4. Llama a la función `mean_ndcg` para cada modelo y compara usando valores de k de 10, 100, 500, 1.000 y 2.000. Analice los resultados y mencione por qué podría producirse la diferencia de valores entre ambos modelos y entre distintos valores de k. Compare también contra lo observado con Zero Proportion @ k.

In [68]:
def mean_ndcg(predictions_dict, k):
    """
    Calcula el nDCG@k promedio sobre todos los usuarios que tienen
    al menos un ítem relevante graduado en el set de testeo.
    """
    ndcgs = []

    for user_id, predictions in predictions_dict.items():
        if user_id not in user_graded_relevance:
            continue
        # Convertir tripletas (user_id, item_id, rating) a pares (item_id, rating)
        rec_list = [(int(item_id), float(rating)) for _, item_id, rating in predictions]
        ndcgs.append(user_ndcg(rec_list, user_graded_relevance[user_id], k))

    return sum(ndcgs) / len(ndcgs) if ndcgs else 0.0


**Respuesta**: -Coloque su respuesta en esta celda-

In [69]:
# El arreglo de Ks que te pidieron en la prueba
ks = [10, 100, 500, 1000, 2000]

print("--- Comparación de mean_ndcg ---")

# El ciclo para imprimir y comparar ambos modelos
for k in ks:
    # Calculamos el nDCG para el modelo basado en usuarios
    ndcg_user = mean_ndcg(user_based_predictions, k)

    # Calculamos el nDCG para el modelo basado en ítems
    ndcg_item = mean_ndcg(item_based_predictions, k)

    # Imprimimos los resultados formateados a 4 decimales
    print(f"K={k:<4} | User-based: {ndcg_user:.4f} | Item-based: {ndcg_item:.4f}")

--- Comparación de mean_ndcg ---
K=10   | User-based: 0.0008 | Item-based: 0.0741
K=100  | User-based: 0.0848 | Item-based: 0.1493
K=500  | User-based: 0.2508 | Item-based: 0.3060
K=1000 | User-based: 0.3095 | Item-based: 0.3481
K=2000 | User-based: 0.3189 | Item-based: 0.3630
